# ChainlinkBlockRelay Test Script

Tests the ChainlinkBlockRelay deployment end-to-end:

1. **Trigger** — HTTP POST to the deployed CRE workflow
2. **Primary relay commit** — poll `committer_votes(relay, block)` on the primary chain; non-zero independent of threshold
3. **CCIP dispatch** — check `BlockHashBroadcast` event confirms the relay sent CCIP messages
4. **Target chain delivery** — check `committer_votes` on CCIP-receiving chains (re-run after ~20 min)

> **Note on threshold**: `committer_votes` succeeds as soon as *this* relay commits, regardless of
> how many other committers (LZBlockRelay, multisig) have committed. `last_confirmed_block_number`
> only advances once `threshold` is reached across all committers.

## 1. Configuration

In [ ]:
%load_ext autoreload
%autoreload 2

NETWORK_TYPE = "mainnets"  # "testnets" or "mainnets"

# Chain where the CRE workflow delivers onReport
CRE_RELAY_CHAIN = "ethereum"

# Chains expected to receive the block via CCIP broadcast from the primary relay.
# Must have ChainlinkBlockRelay deployed and configured as a CCIP peer of CRE_RELAY_CHAIN.
CCIP_TARGET_CHAINS = ["base", "arbitrum"]

# Polling: CRE is off-chain, typically processes within 30-60 s
CRE_POLL_TIMEOUT = 120  # seconds
CRE_POLL_INTERVAL = 10  # seconds

# CRE workflow selector — fill one option and leave the other blank.
# Option A: workflow ID (hex string from CRE dashboard)
CRE_WORKFLOW_ID = ""
# Option B: owner + name + tag (all three required if ID is empty)
CRE_WORKFLOW_OWNER = ""
CRE_WORKFLOW_NAME = ""
CRE_WORKFLOW_TAG = ""

## 2. Environment

In [ ]:
import base64
import hashlib
import json
import logging
import os
import subprocess
import sys
import time
import uuid
from pathlib import Path

import requests
from dotenv import load_dotenv
from eth_account import Account
from eth_account.messages import encode_defunct
from web3 import Web3
from web3.middleware import ExtraDataToPOAMiddleware

sys.path.append(str(Path().resolve().parent))
from DeploymentManager import DeploymentManager

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
load_dotenv()

EMPTY_BYTES32 = b"\x00" * 32

## 3. Helpers

In [ ]:
def get_vyper_abi(filepath):
    result = subprocess.run(
        ["vyper", filepath, "-f", "abi_python"],
        capture_output=True,
        text=True,
        check=True,
    )
    return result.stdout


def poll(fn, timeout, interval, label="waiting"):
    "Call fn() every interval seconds until it returns truthy or timeout expires."
    start = time.time()
    while time.time() - start < timeout:
        result = fn()
        if result:
            return result
        print(f"  [{int(time.time() - start)}s] {label}...")
        time.sleep(interval)
    return None


# ── CRE gateway JWT helpers ───────────────────────────────────────────────────


def _b64url(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).rstrip(b"=").decode()


def _sha256_stable(obj) -> str:
    "SHA-256 of the stable-serialized object (equivalent to json-stable-stringify)."
    s = json.dumps(obj, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(s.encode()).hexdigest()


def create_cre_jwt(jsonrpc_request: dict, private_key: str) -> str:
    "Create a CRE gateway JWT signed with an Ethereum private key."
    account = Account.from_key(private_key)
    header = {"alg": "ETH", "typ": "JWT"}
    now = int(time.time())
    payload = {
        "digest": "0x" + _sha256_stable(jsonrpc_request),
        "iss": account.address,
        "iat": now,
        "exp": now + 300,
        "jti": str(uuid.uuid4()),
    }

    encoded_header = _b64url(json.dumps(header, separators=(",", ":")).encode())
    encoded_payload = _b64url(json.dumps(payload, sort_keys=True, separators=(",", ":")).encode())
    raw_message = f"{encoded_header}.{encoded_payload}"

    # Ethereum personal sign (adds "\x19Ethereum Signed Message:\n" prefix)
    signable = encode_defunct(text=raw_message)
    signed = account.sign_message(signable)

    r = signed.r.to_bytes(32, "big")
    s_bytes = signed.s.to_bytes(32, "big")
    recovery_id = signed.v - 27  # 0 or 1
    sig_bytes = r + s_bytes + bytes([recovery_id])

    return f"{raw_message}.{_b64url(sig_bytes)}"


def trigger_cre_workflow(
    workflow_selector: dict,
    input_data: dict,
    private_key: str,
    gateway_url: str,
):
    "POST a signed JSON-RPC 2.0 request to the CRE gateway."
    request_id = str(uuid.uuid4())
    jsonrpc_request = {
        "jsonrpc": "2.0",
        "id": request_id,
        "method": "workflows.execute",
        "params": {
            "input": input_data,
            "workflow": workflow_selector,
        },
    }
    jwt = create_cre_jwt(jsonrpc_request, private_key)
    # Body must be stable-serialized — same ordering used in the JWT digest
    body = json.dumps(jsonrpc_request, sort_keys=True, separators=(",", ":"))
    return requests.post(
        gateway_url,
        headers={"Content-Type": "application/json", "Authorization": f"Bearer {jwt}"},
        data=body,
        timeout=30,
    )

## 4. Load Deployment State

In [ ]:
deployment_manager = DeploymentManager()
deployed_contracts = deployment_manager.get_all_deployed_contracts(NETWORK_TYPE)
if not deployed_contracts:
    raise ValueError(f"No deployments found for {NETWORK_TYPE}")

with open("../chain-parse/chains.json") as f:
    all_chains_config = json.load(f)[NETWORK_TYPE]

all_test_chains = [CRE_RELAY_CHAIN] + CCIP_TARGET_CHAINS
for chain in all_test_chains:
    contracts = deployed_contracts.get(chain, {})
    if not contracts:
        logging.warning(f"{chain}: no deployments found")
    elif "ChainlinkBlockRelay" not in contracts:
        logging.warning(f"{chain}: ChainlinkBlockRelay not deployed — check deploy.ipynb")
    else:
        logging.info(f"{chain}: ChainlinkBlockRelay at {contracts['ChainlinkBlockRelay']}")

## 5. Initialize Chain State

In [ ]:
ABI_RELAY = get_vyper_abi("../../contracts/messengers/ChainlinkBlockRelay.vy")
ABI_ORACLE = get_vyper_abi("../../contracts/BlockOracle.vy")

ankr_key = os.environ.get("ANKR_API_KEY")
drpc_key = os.environ.get("DRPC_API_KEY")
state_dict = {}

for chain_name in all_test_chains:
    config = all_chains_config.get(chain_name)
    if not config:
        logging.warning(f"No chain config for {chain_name}")
        continue

    rpc_url = None
    for rpc_type in ["ankr", "drpc", "public"]:
        if rpc_type == "public" and NETWORK_TYPE == "testnets":
            rpc_type = "rpc"
        val = config.get(rpc_type)
        if val:
            if rpc_type == "ankr":
                rpc_url = val.format(ankr_key)
            elif rpc_type == "drpc":
                rpc_url = val.format(drpc_key)
            else:
                rpc_url = val
            break

    if not rpc_url:
        logging.warning(f"No RPC for {chain_name}")
        continue

    w3 = Web3(Web3.HTTPProvider(rpc_url))
    w3.middleware_onion.inject(ExtraDataToPOAMiddleware, layer=0)
    contracts = deployed_contracts.get(chain_name, {})

    state_dict[chain_name] = {"w3": w3}

    if "BlockOracle" in contracts:
        state_dict[chain_name]["oracle_w3"] = w3.eth.contract(
            address=contracts["BlockOracle"], abi=ABI_ORACLE
        )
        state_dict[chain_name]["oracle_address"] = contracts["BlockOracle"]

    if "ChainlinkBlockRelay" in contracts:
        state_dict[chain_name]["relay_w3"] = w3.eth.contract(
            address=contracts["ChainlinkBlockRelay"], abi=ABI_RELAY
        )
        state_dict[chain_name]["relay_address"] = contracts["ChainlinkBlockRelay"]

    logging.info(
        f"{chain_name}: connected"
        f" | oracle={'oracle_w3' in state_dict[chain_name]}"
        f" | relay={'relay_w3' in state_dict[chain_name]}"
    )

## 6. Baseline Oracle State

In [ ]:
eth_w3 = state_dict[CRE_RELAY_CHAIN]["w3"]
baseline_eth_block = eth_w3.eth.block_number
print(f"Ethereum block at trigger time: {baseline_eth_block}")
print(f"Will poll committer_votes in range [{baseline_eth_block - 1}, {baseline_eth_block + 5}]\n")

print("Current oracle state:")
for chain_name in all_test_chains:
    s = state_dict.get(chain_name, {})
    if "oracle_w3" not in s:
        print(f"  {chain_name}: oracle not loaded")
        continue
    oracle = s["oracle_w3"]
    relay_addr = s.get("relay_address")
    last = oracle.functions.last_confirmed_block_number().call()
    vote = (
        oracle.functions.committer_votes(relay_addr, last).call()
        if relay_addr and last
        else EMPTY_BYTES32
    )
    print(
        f"  {chain_name}: last_confirmed={last}"
        f" | relay_vote={'set' if vote != EMPTY_BYTES32 else 'none'}"
    )

## 7. Trigger CRE Workflow

In [ ]:
CRE_GATEWAY_URL = os.environ.get("CRE_GATEWAY_URL")
CRE_PRIVATE_KEY = os.environ.get("CRE_PRIVATE_KEY")

if not CRE_GATEWAY_URL:
    raise ValueError(
        "CRE_GATEWAY_URL not set — copy the gateway URL from the CRE workflow dashboard"
    )
if not CRE_PRIVATE_KEY:
    raise ValueError(
        "CRE_PRIVATE_KEY not set — use the private key authorised to trigger this workflow"
    )

# Build workflow selector from config vars (§1)
if CRE_WORKFLOW_ID:
    workflow_selector = {"workflowID": CRE_WORKFLOW_ID}
elif CRE_WORKFLOW_OWNER and CRE_WORKFLOW_NAME and CRE_WORKFLOW_TAG:
    workflow_selector = {
        "workflowOwner": CRE_WORKFLOW_OWNER,
        "workflowName": CRE_WORKFLOW_NAME,
        "workflowTag": CRE_WORKFLOW_TAG,
    }
else:
    raise ValueError(
        "No workflow selector — set CRE_WORKFLOW_ID, or all three of "
        "CRE_WORKFLOW_OWNER + CRE_WORKFLOW_NAME + CRE_WORKFLOW_TAG in §1"
    )

_account = Account.from_key(CRE_PRIVATE_KEY)
logging.info(f"Triggering CRE workflow as {_account.address}")
logging.info(f"  Gateway:  {CRE_GATEWAY_URL}")
logging.info(f"  Selector: {workflow_selector}")

resp = trigger_cre_workflow(workflow_selector, {}, CRE_PRIVATE_KEY, CRE_GATEWAY_URL)
resp.raise_for_status()
trigger_time = time.time()
logging.info(f"CRE workflow triggered at ETH block ~{baseline_eth_block}: HTTP {resp.status_code}")

## 8. Verify Primary Relay Committed

In [ ]:
oracle = state_dict[CRE_RELAY_CHAIN]["oracle_w3"]
relay_addr = state_dict[CRE_RELAY_CHAIN]["relay_address"]

# CRE reads the latest block at workflow execution time — may be a few blocks ahead.
BLOCK_WINDOW = 5


def check_relay_committed():
    for n in range(baseline_eth_block - 1, baseline_eth_block + BLOCK_WINDOW + 1):
        h = oracle.functions.committer_votes(relay_addr, n).call()
        if h != EMPTY_BYTES32:
            return (n, h)
    return None


print(f"Polling committer_votes on {CRE_RELAY_CHAIN} (timeout {CRE_POLL_TIMEOUT}s)...")
result = poll(
    check_relay_committed, CRE_POLL_TIMEOUT, CRE_POLL_INTERVAL, "waiting for relay commit"
)

if result:
    committed_block, committed_hash = result
    print(f"\n✓ ChainlinkBlockRelay committed block {committed_block}: 0x{committed_hash.hex()}")
else:
    committed_block, committed_hash = None, None
    print(
        f"\n✗ No commit in range [{baseline_eth_block - 1}, {baseline_eth_block + BLOCK_WINDOW}]"
        f" within {CRE_POLL_TIMEOUT}s"
    )

## 9. Verify CCIP Broadcast Dispatch

In [ ]:
# BlockHashBroadcast is emitted by ChainlinkBlockRelay when it sends CCIP messages.
# This confirms dispatch without waiting for cross-chain delivery.
if not committed_block:
    print("Skipped — no committed block from §8")
else:
    relay_w3 = state_dict[CRE_RELAY_CHAIN]["relay_w3"]
    scan_from = max(0, eth_w3.eth.block_number - 200)

    try:
        broadcast_logs = relay_w3.events.BlockHashBroadcast.create_filter(
            fromBlock=hex(scan_from)
        ).get_all_entries()

        matching = [log for log in broadcast_logs if log["args"]["block_number"] == committed_block]

        if matching:
            targets = matching[0]["args"]["targets"]
            selectors = [t["chain_selector"] for t in targets]
            print(f"✓ BlockHashBroadcast: CCIP sent to {len(selectors)} chain(s)")
            for t in targets:
                print(f"  selector {t['chain_selector']}, fee {t['fee'] / 1e18:.6f} ETH")
        else:
            print("⚠ BlockHashBroadcast not found for this block")
            print("  Possible causes: no CCIP peers configured, or broadcast was skipped")
    except Exception as e:
        logging.warning(f"Could not fetch BlockHashBroadcast logs: {e}")

    # Oracle threshold check
    confirmed_hash = oracle.functions.get_block_hash(committed_block).call()
    if confirmed_hash != EMPTY_BYTES32:
        print(f"\n✓ Block {committed_block} already confirmed in oracle (threshold reached)")
    else:
        count = oracle.functions.commitment_count(committed_block, committed_hash).call()
        threshold = oracle.functions.threshold().call()
        print(f"\n  Oracle commitments: {count}/{threshold} — threshold not yet reached")
        print("  Expected: other committers (LZBlockRelay, multisig backup) must also commit")

## 10. Verify Target Chain Delivery

CCIP delivery on mainnet takes ~10-20 minutes. Re-run this cell to check progress.

In [ ]:
if not committed_block:
    print("Skipped — no committed block from §8")
else:
    print(f"Checking CCIP delivery for block {committed_block}:\n")
    for chain_name in CCIP_TARGET_CHAINS:
        s = state_dict.get(chain_name, {})
        if "oracle_w3" not in s or "relay_address" not in s:
            print(f"  {chain_name}: not initialized — check §5")
            continue

        target_oracle = s["oracle_w3"]
        target_relay = s["relay_address"]

        vote = target_oracle.functions.committer_votes(target_relay, committed_block).call()
        if vote != EMPTY_BYTES32:
            confirmed = target_oracle.functions.get_block_hash(committed_block).call()
            print(
                f"  ✓ {chain_name}: committed 0x{vote.hex()[:16]}..."
                f" | oracle confirmed: {confirmed != EMPTY_BYTES32}"
            )
        else:
            print(f"  ⏳ {chain_name}: not yet delivered — re-run this cell in a few minutes")

## 11. Summary

In [ ]:
print("\n" + "=" * 72)
print("CHAINLINK RELAY TEST SUMMARY")
print("=" * 72)
print(f"Network:       {NETWORK_TYPE}")
print(f"Primary chain: {CRE_RELAY_CHAIN}")
print(f"ETH baseline:  block {baseline_eth_block}")
print()

if committed_block:
    print(f"✓ CRE workflow delivered block {committed_block} to ChainlinkBlockRelay")
    print(f"  Hash: 0x{committed_hash.hex()}")

    confirmed_hash = oracle.functions.get_block_hash(committed_block).call()
    count = oracle.functions.commitment_count(committed_block, committed_hash).call()
    threshold = oracle.functions.threshold().call()
    status = (
        "CONFIRMED" if confirmed_hash != EMPTY_BYTES32 else f"{count}/{threshold} commits (pending)"
    )
    print(f"  Oracle:  {status}")

    print()
    for chain_name in CCIP_TARGET_CHAINS:
        s = state_dict.get(chain_name, {})
        if "oracle_w3" not in s or "relay_address" not in s:
            print(f"  {chain_name}: not loaded")
            continue
        vote = s["oracle_w3"].functions.committer_votes(s["relay_address"], committed_block).call()
        symbol = "✓" if vote != EMPTY_BYTES32 else "⏳"
        label = "committed via CCIP" if vote != EMPTY_BYTES32 else "pending CCIP delivery"
        print(f"  {symbol} {chain_name}: {label}")
else:
    print("✗ ChainlinkBlockRelay commit was not detected")
    print(
        "  Check: CRE_GATEWAY_URL is correct, CRE_PRIVATE_KEY is authorized, workflow is deployed, relay is configured"
    )